# 04: Feature Representation and Baselines

This notebook builds the shared per-type encoders and the raw/classical baselines, applied identically to whichever candidate is active, using new code in `src/features/` alongside reused utilities from `src/eda/`.

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda import config
from src.eda.report import log_findings, savefig

# src/features/ is currently just an empty package (Prompt 1.3 fills it
# in) -- imported here so Setup already reflects where the encoders/
# baselines built in this notebook will live.
from src import features

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
FINAL_DIR = REPO_ROOT / "data" / "final"
FIGURES_DIR = REPO_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Load active candidate

In [ ]:
CANDIDATES = {
    "br_gt0": {
        "train": FINAL_DIR / "tmdb_br_gt0_train.parquet",
        "val": FINAL_DIR / "tmdb_br_gt0_val.parquet",
        "test": FINAL_DIR / "tmdb_br_gt0_test.parquet",
    },
    "nbr_gt5": {
        "train": FINAL_DIR / "tmdb_nbr_gt5_train.parquet",
        "val": FINAL_DIR / "tmdb_nbr_gt5_val.parquet",
        "test": FINAL_DIR / "tmdb_nbr_gt5_test.parquet",
    },
}

In [ ]:
# Single switch: "br_gt0" is the primary candidate, "nbr_gt5" the
# fallback. Every later cell reads train_df/val_df/test_df (or
# ACTIVE_CANDIDATE for naming saved artifacts) instead of a literal
# candidate name, so switching candidates means editing only this line.
ACTIVE_CANDIDATE = "br_gt0"
# ACTIVE_CANDIDATE = "nbr_gt5"

In [ ]:
train_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["train"])
val_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["val"])
test_df = pd.read_parquet(CANDIDATES[ACTIVE_CANDIDATE]["test"])

print(f"Active candidate: {ACTIVE_CANDIDATE}")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}") 

## 2. Target/feature split

In [ ]:
from src.features.target_split import build_feature_target_split

train_split = build_feature_target_split(train_df)

print(f"target_col: {train_split.target_col}")
print(f"id_cols: {len(train_split.id_cols)}")
print(f"feature_cols: {len(train_split.feature_cols)}")
print(f"first 5 feature_cols: {train_split.feature_cols}")

In [ ]:
val_split = build_feature_target_split(val_df)
test_split = build_feature_target_split(test_df)

assert train_split.feature_cols == val_split.feature_cols == test_split.feature_cols, (
    "feature_cols differ across splits: "
    f"train={train_split.feature_cols}, val={val_split.feature_cols}, test={test_split.feature_cols}"
)
assert train_split.id_cols == val_split.id_cols == test_split.id_cols, (
    "id_cols differ across splits: "
    f"train={train_split.id_cols}, val={val_split.id_cols}, test={test_split.id_cols}"
)

print("Feature/target split consistent across train/val/test")

## 3. Numeric standardisation

In [ ]:
numeric_dtype_cols = set(train_df.select_dtypes(include="number").columns)
NUMERIC_COLS = [
    c for c in train_split.feature_cols
    if c in numeric_dtype_cols and c not in {"release_month_sin", "release_month_cos"}
]
print(NUMERIC_COLS)

Matches: 
- {budget, revenue, runtime, popularity, release_year} for br_gt0
- {runtime, popularity, release_year} for nbr_gt5 (budget/revenue absent)

### Fit on train, transform all three splits

In [ ]:
from src.features.standardize import Standardizer

std = Standardizer(NUMERIC_COLS)
std.fit(train_df)

train_df_std = std.transform(train_df)
val_df_std = std.transform(val_df)
test_df_std = std.transform(test_df)

print("train mean:")
print(train_df_std[NUMERIC_COLS].mean())
print("train std:")
print(train_df_std[NUMERIC_COLS].std())

print("val mean:")
print(val_df_std[NUMERIC_COLS].mean())
print("val std:")
print(val_df_std[NUMERIC_COLS].std())

### Unit check: manual z-score vs. `Standardizer` output

In [ ]:
manual_runtime_std = (train_df["runtime"] - train_df["runtime"].mean()) / train_df["runtime"].std(ddof=0)
class_runtime_std = train_df_std["runtime"]

print(f"Exact match: {manual_runtime_std.equals(class_runtime_std)}")
print(f"Max absolute difference: {(manual_runtime_std - class_runtime_std).abs().max()}")

### Persist the fitted `Standardizer`

In [ ]:
import json

ARTIFACTS_DIR = REPO_ROOT / "data" / "artifacts" / ACTIVE_CANDIDATE
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

standardizer_path = ARTIFACTS_DIR / "standardizer.json"
with open(standardizer_path, "w") as f:
    json.dump(std.to_dict(), f)
print(f"Saved: {standardizer_path}")

with open(standardizer_path) as f:
    std_reloaded = Standardizer.from_dict(json.load(f))

assert std_reloaded.mean_.equals(std.mean_), "reloaded mean_ does not match the original"
assert std_reloaded.std_.equals(std.std_), "reloaded std_ does not match the original"
print("Round-trip check passed: reloaded mean_/std_ match the original exactly")